In [ ]:
import pandas as pd
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from node2vec import Node2Vec
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# -----------------------------
# 1. Load edge list dataset
# -----------------------------
# Change this filename to your dataset file
file_path = "Datasets/facebook_combined.txt"

df = pd.read_csv(file_path)

# Expect columns like: source, target
source_col = df.columns[0]
target_col = df.columns[1]

G = nx.from_pandas_edgelist(df, source=source_col, target=target_col, create_using=nx.Graph())

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

# -----------------------------
# 2. Graph mining features
# -----------------------------
degree_centrality = nx.degree_centrality(G)
betweenness_centrality = nx.betweenness_centrality(G, k=min(500, G.number_of_nodes()), seed=42)
closeness_centrality = nx.closeness_centrality(G)
pagerank = nx.pagerank(G)

features = []
for node in G.nodes():
    features.append([
        node,
        degree_centrality.get(node, 0),
        betweenness_centrality.get(node, 0),
        closeness_centrality.get(node, 0),
        pagerank.get(node, 0),
        G.degree(node),
        nx.clustering(G, node)
    ])

feat_df = pd.DataFrame(features, columns=[
    "node", "degree_centrality", "betweenness_centrality",
    "closeness_centrality", "pagerank", "degree", "clustering"
])

# -----------------------------
# 3. Create pseudo-labels
# -----------------------------
# If you do not have ground-truth fraud labels, use a simple anomaly rule.
# Example: nodes with very high degree or low clustering can be suspicious.
deg_thresh = feat_df["degree"].quantile(0.95)
cluster_thresh = feat_df["clustering"].quantile(0.10)

feat_df["label"] = ((feat_df["degree"] >= deg_thresh) | (feat_df["clustering"] <= cluster_thresh)).astype(int)

# -----------------------------
# 4. Node2Vec embeddings
# -----------------------------
node2vec = Node2Vec(G, dimensions=64, walk_length=30, num_walks=100, workers=2, seed=42)
n2v_model = node2vec.fit(window=10, min_count=1, batch_words=4)

embedding_df = pd.DataFrame(
    [n2v_model.wv[str(node)] if str(node) in n2v_model.wv else np.zeros(64) for node in feat_df["node"]],
    columns=[f"emb_{i}" for i in range(64)]
)

data = pd.concat([feat_df.reset_index(drop=True), embedding_df], axis=1)

# -----------------------------
# 5. Prepare data for ML
# -----------------------------
X = data.drop(columns=["node", "label"])
y = data["label"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

# -----------------------------
# 6. Train classifier
# -----------------------------
clf = RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced")
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc = roc_auc_score(y_test, y_prob)

print("Accuracy:", acc)
print("F1 Score:", f1)
print("ROC-AUC:", roc)
print(classification_report(y_test, y_pred))

# -----------------------------
# 7. Save outputs
# -----------------------------
data.to_csv("facebook_graph_features.csv", index=False)

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=300)
plt.close()

# -----------------------------
# 8. Graph visualization
# -----------------------------
plt.figure(figsize=(10, 8))
pos = nx.spring_layout(G.subgraph(list(G.nodes())[:300]), seed=42)
nx.draw(G.subgraph(list(G.nodes())[:300]), pos, node_size=20, width=0.3, alpha=0.7)
plt.title("Facebook Subgraph Visualization")
plt.tight_layout()
plt.savefig("facebook_subgraph.png", dpi=300)
plt.close()

# -----------------------------
# 9. Embedding visualization
# -----------------------------
pca = PCA(n_components=2, random_state=42)
emb_2d = pca.fit_transform(data[[f"emb_{i}" for i in range(64)]].values)

plt.figure(figsize=(8, 6))
plt.scatter(emb_2d[:, 0], emb_2d[:, 1], c=data["label"], cmap="coolwarm", s=15)
plt.title("Node2Vec Embedding Visualization")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.tight_layout()
plt.savefig("node2vec_pca.png", dpi=300)
plt.close()

Nodes: 0
Edges: 0


Computing transition probabilities: 0it [00:00, ?it/s]

Generating walks (CPU: 1): 100%|██████████| 40/40 [00:00<?, ?it/s]


RuntimeError: you must first build vocabulary before training the model